# Machine-learning model comparison

This notebook compares linear regression, Ridge regression, Random Forest,
gradient boosting, Extra Trees, and XGBoost using bond counts, RDKit
descriptors, Morgan fingerprints, and Mordred/ChemML descriptors.

The principal evaluation uses a fixed 80/20 split with `random_state=42`.


In [ ]:
import numpy as np
import pandas as pd
import re

import matplotlib.pyplot as plt
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn import linear_model
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor



## 1. Load the processed dataset


In [ ]:
filename = "../data/ATcT_bonds_corrected.csv"

df = pd.read_csv(filename)

print(df.shape)
df.head()


## 2. Data checks and feature matrix


In [ ]:
duplicate_like = [
    col for col in df.columns
    if re.search(r"\.\d+$", str(col))
]

print("Remaining duplicate-like columns:")
print(duplicate_like)

df = (
    df.drop_duplicates()
    .dropna(
        subset=[
            "species_name",
            "dHf_298K",
            "Canonical_SMILES",
        ]
    )
    .reset_index(drop=True)
)

print("Usable rows:", len(df))


In [ ]:
df.describe(include="all")

In [ ]:
y = df["dHf_298K"]
exclude = [
    "species_name",
    "formula",
    "dHf_0K",
    "dHf_298K",
    "units",
    "atct_id",
    "molar_mass_uncertainty",
    "cas_number",
    "atct_seq",
    "Canonical_SMILES",
    "Isomeric_SMILES",
    "InChI",
    "Structure_source",
    'uncertainty_kJmol', 
    "CID",
    "cleaned_formula",
    "species_type"
]

X = df.drop(columns=exclude)

print(X.shape)
print(y.shape)

In [ ]:
print(X.select_dtypes(exclude="number").columns)

In [ ]:
constant_cols = X.columns[X.nunique() == 1]

print(f"Removing {len(constant_cols)} constant descriptors:")
print(constant_cols.tolist())

X = X.drop(columns=constant_cols)

## 3. Exploratory target and correlation analysis


In [ ]:
target = "dHf_298K"

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Scatter
axes[0].scatter(df["molar_mass"],
                df[target],
                s=15,
                alpha=0.7)

axes[0].set_xlabel("Molar mass (g/mol)")
axes[0].set_ylabel(r"$\Delta H_f^\circ(298\,K)$ (kJ/mol)")
axes[0].grid(alpha=0.3)

# Histogram
axes[1].hist(df[target], bins=100, edgecolor="black")

axes[1].set_xlabel(r"$\Delta H_f^\circ(298\,K)$ (kJ/mol)")
axes[1].set_ylabel("Count")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
Xy = pd.concat([X,y], axis = 1)
print(Xy.shape)

Xy_corr = Xy.corr(method='pearson')
print(Xy_corr.shape)
Xy_corr.tail(n = 10)

In [ ]:
dHf_corr = Xy_corr["dHf_298K"].sort_values(
    ascending=False
)

print(dHf_corr)

In [ ]:
corr = Xy_corr.dropna()

In [ ]:
corr_values = (
    corr["dHf_298K"]
    .drop("dHf_298K")
    .dropna()
)

positive = corr_values.sort_values(ascending=False).head(20)
negative = corr_values.sort_values(ascending=True).head(20)

print("Positive correlations")
print(positive)

print("\nNegative correlations")
print(negative)

In [ ]:
top_corr = pd.concat([positive, negative])

# Remove duplicated features if any
top_corr = top_corr[~top_corr.index.duplicated()]

# Sort for plotting
top_corr = top_corr.sort_values()

plt.figure(figsize=(10,12))

top_corr.plot(
    kind="barh"
)

plt.axvline(0, linewidth=1)

plt.xlabel("Pearson correlation with dHf_298K")
plt.ylabel("Bond feature")

plt.title("Top positive and negative correlations with ΔHf(298 K)")

plt.tight_layout()
plt.show()

In [ ]:
correlations = (
    corr["dHf_298K"]
    .drop("dHf_298K")
    .sort_values()
)

plt.figure(figsize=(8,10))

correlations.plot(kind="barh")

plt.xlabel("Pearson correlation")
plt.ylabel("Feature")

plt.tight_layout()
plt.show()

In [ ]:
top = (
    correlations.abs()
    .sort_values(ascending=False)
    .head(6)
    .index
)

top

In [ ]:
spearman = Xy.corr(method="spearman")

spearman["dHf_298K"].sort_values()

In [ ]:
fig, axes = plt.subplots(2,3, figsize=(12,8))

for ax, feature in zip(axes.ravel(), top):

    ax.scatter(df[feature], df["dHf_298K"], s=8)

    ax.set_xlabel(feature)
    ax.set_ylabel("ΔHf°298")

plt.tight_layout()
plt.show()

## 4. Linear-regression baselines

### 4.1 All molecular species


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)


In [ ]:
ols_model = LinearRegression()

ols_model.fit(X_train, y_train)

In [ ]:
y_pred = ols_model.predict(X_test)

In [ ]:
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"RMSE: {rmse:.3f} kJ/mol")
print(f"MAE:  {mae:.3f} kJ/mol")
print(f"R²:   {r2:.3f}")


In [ ]:
plt.figure(figsize=(6,6))

plt.scatter(y_test, y_pred, s=15, alpha=0.6)

plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    '--'
)

plt.xlabel(r"Actual $\Delta H_f^\circ(298K)$ (kJ/mol)")
plt.ylabel(r"Predicted $\Delta H_f^\circ(298K)$ (kJ/mol)")

plt.tight_layout()
plt.show()

In [ ]:
coefficients = pd.DataFrame({
    "descriptor": X.columns,
    "coefficient": ols_model.coef_
})

coefficients.sort_values(
    by="coefficient",
    ascending=False
).head(10)

In [ ]:
coefficients.sort_values(
    by="coefficient",
    ascending=True
).head(10)

### 4.2 Neutral and ionic subsets


In [ ]:
df["Charge"].value_counts()

In [ ]:
neutral_df = df[df["Charge"] == 0]
ion_df = df[df["Charge"] != 0]

print("Neutral molecules:", neutral_df.shape[0])
print("Ions:", ion_df.shape[0])

In [ ]:
descriptor_columns = X.columns

X_neutral = neutral_df[descriptor_columns]
y_neutral = neutral_df["dHf_298K"]

X_ion = ion_df[descriptor_columns]
y_ion = ion_df["dHf_298K"]

In [ ]:
def evaluate_ols(X, y):

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        random_state=42
    )

    model = LinearRegression()
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    return rmse, mae, r2


print("Neutral molecules")
print(evaluate_ols(X_neutral, y_neutral))

print("\nIons")
print(evaluate_ols(X_ion, y_ion))

In [ ]:
plt.figure(figsize=(8,5))

plt.hist(
    neutral_df["dHf_298K"],
    bins=80,
    alpha=0.6,
    label="Neutral"
)

plt.hist(
    ion_df["dHf_298K"],
    bins=80,
    alpha=0.6,
    label="Ions"
)

plt.xlabel(r"$\Delta H_f^\circ(298K)$ (kJ/mol)")
plt.ylabel("Count")
plt.legend()

plt.show()

### 4.3 Include molecular charge


In [ ]:
X_charge = X.copy()

X_charge["Charge"] = df["Charge"]

In [ ]:
print(X_charge.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_charge,
    y,
    test_size=0.20,
    random_state=42
)

In [ ]:
ols_charge = LinearRegression()

ols_charge.fit(
    X_train,
    y_train
)

In [ ]:
y_pred = ols_charge.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"RMSE: {rmse:.3f} kJ/mol")
print(f"MAE:  {mae:.3f} kJ/mol")
print(f"R²:   {r2:.3f}")

In [ ]:
coef = pd.DataFrame({
    "feature": X_charge.columns,
    "coefficient": ols_charge.coef_
})

coef[coef["feature"]=="Charge"]

In [ ]:
charge_index = list(X_charge.columns).index("Charge")

print(ols_charge.coef_[charge_index])

In [ ]:
test_indices = X_test.index

neutral_test = df.loc[test_indices, "Charge"] == 0
ion_test = df.loc[test_indices, "Charge"] != 0

print("Neutral RMSE:")
print(
    np.sqrt(
        mean_squared_error(
            y_test[neutral_test],
            y_pred[neutral_test]
        )
    )
)

print("Ion RMSE:")
print(
    np.sqrt(
        mean_squared_error(
            y_test[ion_test],
            y_pred[ion_test]
        )
    )
)

### 4.4 Include molar mass and charge


In [ ]:
X_mass = X.copy()

X_mass["molar_mass"] = df["molar_mass"]

In [ ]:
plt.figure(figsize=(6,6))

plt.scatter(y_test, y_pred, s=15, alpha=0.5)

plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    "--"
)

plt.xlabel("Actual")
plt.ylabel("Predicted")

plt.show()

In [ ]:
print("Actual:")
print(y_test.describe())

print("\nPredicted:")
print(pd.Series(y_pred).describe())

In [ ]:
np.corrcoef(y_test, y_pred)[0,1]

In [ ]:
r2_score(y_test, y_pred)

In [ ]:
residuals = y_test - y_pred

plt.figure(figsize=(7,5))

plt.scatter(y_pred, residuals, s=15)

plt.axhline(0, linestyle="--")

plt.xlabel("Predicted ΔHf (kJ/mol)")
plt.ylabel("Residual (kJ/mol)")

plt.show()

In [ ]:
X_mass_charge = X.copy()

X_mass_charge["molar_mass"] = df["molar_mass"]
X_mass_charge["Charge"] = df["Charge"]

In [ ]:
print(X_mass_charge.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_mass_charge,
    y,
    test_size=0.20,
    random_state=42
)

In [ ]:
ols_mass_charge = LinearRegression()

ols_mass_charge.fit(
    X_train,
    y_train
)

In [ ]:
y_pred = ols_mass_charge.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"RMSE: {rmse:.3f} kJ/mol")
print(f"MAE:  {mae:.3f} kJ/mol")
print(f"R²:   {r2:.3f}")

In [ ]:
plt.figure(figsize=(6,6))

plt.scatter(y_test, y_pred, s=15, alpha=0.5)

plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    "--"
)

plt.xlabel("Actual")
plt.ylabel("Predicted")

plt.show()

In [ ]:
plt.scatter(y_test, y_pred)

In [ ]:
residuals = y_test - y_pred

plt.figure(figsize=(7,5))

plt.scatter(y_pred, residuals, s=15)

plt.axhline(0, linestyle="--")

plt.xlabel("Predicted ΔHf (kJ/mol)")
plt.ylabel("Residual (kJ/mol)")

plt.show()

## 5. Correlation filtering and regularized linear models


In [ ]:
y = df["dHf_298K"]
exclude = [
    "species_name",
    "formula",
    "dHf_0K",
    "dHf_298K",
    "units",
    "atct_id",
    "molar_mass_uncertainty",
    "cas_number",
    "atct_seq",
    "Canonical_SMILES",
    "Isomeric_SMILES",
    "InChI",
    "Structure_source",
    'uncertainty_kJmol', 
    'bond_difference',
    "CID"
]

X = df.drop(columns=exclude)

print(X.shape)
print(y.shape)

In [ ]:
corr_matrix = X.corr().abs()

upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

to_drop = [
    column
    for column in upper.columns
    if any(upper[column] > 0.95)
]

X_corr = X.drop(columns=to_drop)

print(f"Removed {len(to_drop)} correlated descriptors.")
print(f"Remaining descriptors: {X_corr.shape[1]}")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_corr,
    y,
    test_size=0.20,
    random_state=42
)

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
ols = LinearRegression()

ols.fit(X_train_scaled, y_train)

y_pred_ols = ols.predict(X_test_scaled)

In [ ]:
from sklearn.linear_model import RidgeCV

alphas = np.logspace(-4,4,100)

ridge = RidgeCV(
    alphas=alphas,
    cv=5
)

ridge.fit(
    X_train_scaled,
    y_train
)

print("Best alpha:", ridge.alpha_)

In [ ]:
y_pred_ridge = ridge.predict(X_test_scaled)

In [ ]:
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

def evaluate(y_true, y_pred):

    rmse = np.sqrt(mean_squared_error(y_true,y_pred))
    mae = mean_absolute_error(y_true,y_pred)
    r2 = r2_score(y_true,y_pred)

    return rmse, mae, r2

In [ ]:
ols_metrics = evaluate(y_test,y_pred_ols)
ridge_metrics = evaluate(y_test,y_pred_ridge)

print("OLS")
print(ols_metrics)

print()

print("Ridge")
print(ridge_metrics)

In [ ]:
results = pd.DataFrame(
    {
        "RMSE":[ols_metrics[0],ridge_metrics[0]],
        "MAE":[ols_metrics[1],ridge_metrics[1]],
        "R²":[ols_metrics[2],ridge_metrics[2]]
    },
    index=["OLS","Ridge"]
)

results

## 6. Tree-based ensemble models


In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=500,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train,y_train)

y_pred_rf = rf.predict(X_test)

rf_metrics = evaluate(y_test,y_pred_rf)

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

gbr = GradientBoostingRegressor(
    random_state=42
)

gbr.fit(X_train,y_train)

y_pred_gbr = gbr.predict(X_test)

gbr_metrics = evaluate(y_test,y_pred_gbr)

In [ ]:
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

high_corr = (
    upper.stack()
         .reset_index()
)

high_corr.columns = ["Descriptor 1","Descriptor 2","Correlation"]

high_corr = high_corr[
    high_corr["Correlation"] > 0.95
]

print(high_corr.shape)
high_corr.head(20)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=500,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

In [ ]:
y_pred_rf = rf.predict(X_test)

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

rmse = np.sqrt(mean_squared_error(y_test, y_pred_rf))
mae = mean_absolute_error(y_test, y_pred_rf)
r2 = r2_score(y_test, y_pred_rf)

print(f"RMSE: {rmse:.3f} kJ/mol")
print(f"MAE:  {mae:.3f} kJ/mol")
print(f"R²:   {r2:.3f}")

In [ ]:
results.loc["Random Forest"] = [rmse, mae, r2]
results

In [ ]:
plt.figure(figsize=(6,6))

plt.scatter(y_test, y_pred_rf,
            s=15,
            alpha=0.6)

plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    "r--",
    lw=2
)

plt.xlabel("Actual ΔHf (kJ/mol)")
plt.ylabel("Predicted ΔHf (kJ/mol)")
plt.title("Random Forest")

plt.tight_layout()
plt.show()

In [ ]:
importance = pd.DataFrame({
    "Descriptor": X.columns,
    "Importance": rf.feature_importances_
})

importance = importance.sort_values(
    "Importance",
    ascending=False
)

importance.head(20)

In [ ]:
plt.figure(figsize=(8,6))

plt.barh(
    importance["Descriptor"][:20][::-1],
    importance["Importance"][:20][::-1]
)

plt.xlabel("Importance")
plt.tight_layout()
plt.show()

### 6.1 Neutral and ionic subsets


In [ ]:
neutral_df = df[df["Charge"] == 0].copy()
ion_df = df[df["Charge"] != 0].copy()

In [ ]:
descriptor_cols = X.columns

X_neutral = neutral_df[descriptor_cols]
y_neutral = neutral_df["dHf_298K"]

X_ion = ion_df[descriptor_cols]
y_ion = ion_df["dHf_298K"]

In [ ]:
def evaluate_random_forest(X, y):

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42
    )

    rf = RandomForestRegressor(
        n_estimators=1000,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        max_features="sqrt",
        bootstrap=True,
        random_state=42,
        n_jobs=-1
    )

    rf.fit(X_train, y_train)

    y_pred = rf.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    return {
    "model": rf,
    "y_test": y_test,
    "y_pred": y_pred,
    "rmse": rmse,
    "mae": mae,
    "r2": r2,
}

In [ ]:
neutral_metrics = evaluate_random_forest(X_neutral, y_neutral)
ion_metrics = evaluate_random_forest(X_ion, y_ion)

print("Neutral:", neutral_metrics)
print("Ions:", ion_metrics)

In [ ]:
results = pd.DataFrame(
    columns=["RMSE", "MAE", "R2"]
)
results.loc["RF (Neutral)"] = [
    neutral_metrics["rmse"],
    neutral_metrics["mae"],
    neutral_metrics["r2"],
]

results.loc["RF (Ions)"] = [
    ion_metrics["rmse"],
    ion_metrics["mae"],
    ion_metrics["r2"],
]

results

In [ ]:
rf_all = evaluate_random_forest(X, y)
rf_neutral = evaluate_random_forest(X_neutral, y_neutral)
rf_ion = evaluate_random_forest(X_ion, y_ion)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharex=False, sharey=False)

models = [
    ("All Molecules", rf_all),
    ("Neutral Molecules", rf_neutral),
    ("Ions", rf_ion)
]

for ax, (title, result) in zip(axes, models):

    y_true = result["y_test"]
    y_pred = result["y_pred"]

    ax.scatter(y_true, y_pred,
               s=15,
               alpha=0.6)

    # Perfect prediction line
    all_true = np.concatenate([
        rf_all["y_test"],
        rf_neutral["y_test"],
        rf_ion["y_test"]])

    all_pred = np.concatenate([
        rf_all["y_pred"],
        rf_neutral["y_pred"],
        rf_ion["y_pred"]])

    mn = min(all_true.min(), all_pred.min())
    mx = max(all_true.max(), all_pred.max())

    ax.plot([mn, mx], [mn, mx], 'r--', lw=2)
    ax.set_xlim(mn, mx)
    ax.set_ylim(mn, mx)

    ax.set_title(
        f"{title}\n"
        f"RMSE = {result['rmse']:.1f}\n"
        f"$R^2$ = {result['r2']:.3f}"
    )

    ax.set_xlabel(r"Actual $\Delta H_f^\circ$ (kJ/mol)")
    ax.grid(alpha=0.3)

axes[0].set_ylabel(r"Predicted $\Delta H_f^\circ$ (kJ/mol)")

plt.tight_layout()
plt.show()

### 6.2 Random Forest hyperparameter search


In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor

In [ ]:
param_dist = {
    "n_estimators": [300, 500, 800, 1000, 1500],
    "max_depth": [None, 10, 20, 30, 40, 60],
    "max_features": ["sqrt", "log2", 0.3, 0.5],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "bootstrap": [True, False]
}

In [ ]:
rf = RandomForestRegressor(
    random_state=42,
    n_jobs=-1,
    oob_score=False
)

search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=50,
    cv=5,
    scoring="neg_root_mean_squared_error",
    random_state=42,
    n_jobs=-1,
    verbose=2
)

search.fit(X_train, y_train)

In [ ]:
search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=50,
    cv=5,
    scoring="neg_root_mean_squared_error",
    random_state=42,
    n_jobs=-1,
    verbose=2
)

search.fit(X_train, y_train)

In [ ]:
print(search.best_params_)
print("Best CV RMSE:", -search.best_score_)

In [ ]:
best_rf = search.best_estimator_

y_pred = best_rf.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(rmse, mae, r2)

In [ ]:
residuals = y_test - y_pred

plt.figure(figsize=(7,5))

plt.scatter(
    y_pred,
    residuals,
    s=15,
    alpha=0.5
)

plt.axhline(0, linestyle="--")

plt.xlabel("Predicted ΔHf (kJ/mol)")
plt.ylabel("Residual (kJ/mol)")

plt.tight_layout()
plt.show()

In [ ]:
errors = pd.DataFrame({
    "Actual": y_test,
    "Predicted": y_pred,
    "Error": y_test - y_pred,
    "Abs_Error": np.abs(y_test-y_pred)
})

errors = errors.sort_values(
    "Abs_Error",
    ascending=False
)

errors.head(20)

In [ ]:
worst_idx = errors.head(20).index

df.loc[worst_idx, [
    "species_name",
    "formula",
    "Charge",
    "dHf_298K"
]]

## 7. RDKit descriptor models


In [ ]:
from rdkit.Chem import Descriptors
from rdkit import Chem

In [ ]:
descriptor_names = [name for name, _ in Descriptors._descList]

def rdkit_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return [None] * len(descriptor_names)

    values = []
    for _, func in Descriptors._descList:
        try:
            values.append(func(mol))
        except:
            values.append(None)

    return values

In [ ]:
rdkit_df = df["Canonical_SMILES"].apply(rdkit_descriptors)

rdkit_df = pd.DataFrame(
    rdkit_df.tolist(),
    columns=descriptor_names
)

In [ ]:
rdkit_df.shape

In [ ]:
rdkit_df = rdkit_df.dropna(axis=1)
rdkit_df = rdkit_df.loc[:, rdkit_df.nunique() > 1]

In [ ]:
X_new = pd.concat(
    [
        X.reset_index(drop=True),
        rdkit_df.reset_index(drop=True)
    ],
    axis=1
)

In [ ]:
print(X.shape)
print(rdkit_df.shape)
print(X_new.shape)

In [ ]:
mol = Chem.MolFromSmiles(df["Canonical_SMILES"].iloc[0])

print(df["Canonical_SMILES"].iloc[0])
print(mol)

In [ ]:
invalid = 0

for smi in df["Canonical_SMILES"]:
    if Chem.MolFromSmiles(str(smi)) is None:
        invalid += 1

print(invalid)

In [ ]:
df["Canonical_SMILES"].head(10)

In [ ]:
print(df["Canonical_SMILES"].isna().sum())

In [ ]:
print(df["Canonical_SMILES"].dtype)
df["Canonical_SMILES"].head(10)

In [ ]:
test = rdkit_descriptors(df["Canonical_SMILES"].iloc[0])

print(len(test))
print(test[:10])

In [ ]:
def basic_rdkit_descriptors(smiles):

    mol = Chem.MolFromSmiles(str(smiles))

    if mol is None:
        return {
            "MolWt": np.nan,
            "ExactMolWt": np.nan,
            "HeavyAtomCount": np.nan,
            "NumValenceElectrons": np.nan,
            "TPSA": np.nan,
            "MolLogP": np.nan,
            "NumHDonors": np.nan,
            "NumHAcceptors": np.nan,
            "NumRotatableBonds": np.nan,
            "RingCount": np.nan,
            "FractionCSP3": np.nan,
            "NumAromaticRings": np.nan,
        }

    return {
        "MolWt": Descriptors.MolWt(mol),
        "ExactMolWt": Descriptors.ExactMolWt(mol),
        "HeavyAtomCount": Descriptors.HeavyAtomCount(mol),
        "NumValenceElectrons": Descriptors.NumValenceElectrons(mol),
        "TPSA": Descriptors.TPSA(mol),
        "MolLogP": Descriptors.MolLogP(mol),
        "NumHDonors": Descriptors.NumHDonors(mol),
        "NumHAcceptors": Descriptors.NumHAcceptors(mol),
        "NumRotatableBonds": Descriptors.NumRotatableBonds(mol),
        "RingCount": Descriptors.RingCount(mol),
        "FractionCSP3": Descriptors.FractionCSP3(mol),
        "NumAromaticRings": Descriptors.NumAromaticRings(mol),
    }

In [ ]:
rdkit_df = df["Canonical_SMILES"].apply(basic_rdkit_descriptors)
rdkit_df = pd.DataFrame(rdkit_df.tolist())

In [ ]:
invalid = []

for i, smi in enumerate(df["Canonical_SMILES"]):
    if Chem.MolFromSmiles(str(smi)) is None:
        invalid.append(i)

print(invalid)

In [ ]:
df.loc[
    [368, 474, 779, 814, 826, 1153, 1192, 1208,
     1528, 1553, 1933, 2051, 2077, 2509,
     2522, 2666, 3214],
    ["species_name", "formula", "Canonical_SMILES"]
]

In [ ]:
invalid = [
    368, 474, 779, 814, 826, 1153, 1192, 1208,
    1528, 1553, 1933, 2051, 2077,
    2509, 2522, 2666, 3214
]

df_clean = df.drop(index=invalid).reset_index(drop=True)

In [ ]:
print(df.shape)
print(df_clean.shape)

In [ ]:
X_clean = X.drop(index=invalid).reset_index(drop=True)

y_clean = (
    df_clean["dHf_298K"]
)

In [ ]:
rdkit_df = (
    df_clean["Canonical_SMILES"]
    .apply(basic_rdkit_descriptors)
)

rdkit_df = pd.DataFrame(rdkit_df.tolist())

In [ ]:
print(rdkit_df.shape)

In [ ]:
X_combined = pd.concat(
    [
        X_clean,
        rdkit_df
    ],
    axis=1
)

In [ ]:
print(X_clean.shape)
print(rdkit_df.shape)
print(X_combined.shape)

In [ ]:
print(rdkit_df.shape)
print(rdkit_df.isna().sum())

In [ ]:
overlap = set(X_clean.columns).intersection(rdkit_df.columns)
print(overlap)

In [ ]:
X_model2 = pd.concat(
    [X_clean, rdkit_df],
    axis=1
)

In [ ]:
print("Bond descriptors :", X_clean.shape)
print("RDKit descriptors:", rdkit_df.shape)
print("Combined         :", X_model2.shape)

In [ ]:
rf_model2 = evaluate_random_forest(
    X_model2,
    y_clean
)

In [ ]:
print(f"RMSE = {rf_model2['rmse']:.3f}")
print(f"MAE  = {rf_model2['mae']:.3f}")
print(f"R²   = {rf_model2['r2']:.3f}")

In [ ]:
comparison = pd.DataFrame({
    "Model": [
        "RF (Bond descriptors)",
        "RF (Bond + RDKit)"
    ],
    "RMSE": [
        rf_all["rmse"],
        rf_model2["rmse"]
    ],
    "MAE": [
        rf_all["mae"],
        rf_model2["mae"]
    ],
    "R²": [
        rf_all["r2"],
        rf_model2["r2"]
    ]
})

comparison

In [ ]:
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf_all["model"].feature_importances_
})

importance = importance.sort_values(
    "Importance",
    ascending=False
)

print(importance.head(20))

In [ ]:
import rdkit
print(rdkit.__version__)

## 8. Morgan-fingerprint models


In [ ]:
from rdkit.Chem import AllChem
from rdkit.Chem import rdFingerprintGenerator

morgan_generator = rdFingerprintGenerator.GetMorganGenerator(
    radius=2,
    fpSize=2048
)

In [ ]:
def morgan_fp(smiles):

    mol = Chem.MolFromSmiles(str(smiles))

    if mol is None:
        return np.full(2048, np.nan)

    fp = morgan_generator.GetFingerprint(mol)

    return np.array(fp)

In [ ]:
fp_list = df_clean["Canonical_SMILES"].apply(morgan_fp)

fp_df = pd.DataFrame(fp_list.tolist())

In [ ]:
print(fp_df.shape)

In [ ]:
fp_df = fp_df.loc[:, fp_df.nunique() > 1]

In [ ]:
print(fp_df.shape)

In [ ]:
X_clean.shape

In [ ]:
X_morgan = pd.concat(
    [
        X_clean.reset_index(drop=True),
        fp_df.reset_index(drop=True)
    ],
    axis=1
)

In [ ]:
X_morgan.columns = X_morgan.columns.astype(str)

In [ ]:
print(X_morgan.columns[:10])


In [ ]:
print(type(X_morgan.columns[0]))

In [ ]:
rf_morgan = evaluate_random_forest(
    X_morgan,
    y_clean
)

In [ ]:
print(X_morgan.shape)

In [ ]:
comparison = pd.DataFrame({
    "Model": [
        "RF (Bond descriptors)",
        "RF (Bond + RDKit)",
        "RF (Bond + Morgan)"
    ],
    "RMSE": [
        rf_all["rmse"],
        rf_model2["rmse"],
        rf_morgan["rmse"]
    ],
    "MAE": [
        rf_all["mae"],
        rf_model2["mae"],
        rf_morgan["mae"]
    ],
    "R2": [
        rf_all["r2"],
        rf_model2["r2"],
        rf_morgan["r2"]
    ]
})

comparison

In [ ]:
def evaluate_random_forest_morgan(X, y, max_features):

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42
    )

    rf = RandomForestRegressor(
        n_estimators=1000,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        max_features=max_features,
        bootstrap=True,
        random_state=42,
        n_jobs=-1
    )
    rf.fit(X_train, y_train)

    y_pred = rf.predict(X_test)

    return {
        "model": rf,
        "y_test": y_test,
        "y_pred": y_pred,
        "rmse": np.sqrt(mean_squared_error(y_test, y_pred)),
        "mae": mean_absolute_error(y_test, y_pred),
        "r2": r2_score(y_test, y_pred)
    }

In [ ]:
results = {}

for mf in ["sqrt", 0.2,0.4,0.6,0.8,1.0]:

    results[mf] = evaluate_random_forest_morgan(
        X_morgan,
        y_clean,
        max_features=mf
    )

    print(
        mf,
        results[mf]["rmse"],
        results[mf]["r2"]
    )

In [ ]:
def evaluate_random_forest_morgan(X, y):

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42
    )

    rf = RandomForestRegressor(
        n_estimators=1000,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        max_features=1.0,
        bootstrap=True,
        random_state=42,
        n_jobs=-1
    )

    rf.fit(X_train, y_train)

    y_pred = rf.predict(X_test)

    return {
        "model": rf,
        "y_test": y_test,
        "y_pred": y_pred,
        "rmse": np.sqrt(mean_squared_error(y_test, y_pred)),
        "mae": mean_absolute_error(y_test, y_pred),
        "r2": r2_score(y_test, y_pred)
    }

In [ ]:
rf_morgan_only = evaluate_random_forest_morgan(
    fp_df,
    y_clean
)

In [ ]:
print(f"RMSE: {rf_morgan_only['rmse']:.3f}")
print(f"MAE:  {rf_morgan_only['mae']:.3f}")
print(f"R²:   {rf_morgan_only['r2']:.3f}")

In [ ]:
comparison = pd.DataFrame({
    "Model": [
        "RF Bond descriptors",
        "RF Morgan fingerprints",
        "RF Bond + Morgan"
    ],
    "RMSE": [
        rf_all["rmse"],
        rf_morgan_only["rmse"],
        rf_morgan["rmse"]
    ],
    "MAE": [
        rf_all["mae"],
        rf_morgan_only["mae"],
        rf_morgan["mae"]
    ],
    "R2": [
        rf_all["r2"],
        rf_morgan_only["r2"],
        rf_morgan["r2"]
    ]
})

comparison

In [ ]:
extra_features = df_clean[
    [
        "molar_mass",
        "Charge"
    ]
].reset_index(drop=True)

In [ ]:
print(extra_features.head())

In [ ]:
from rdkit import Chem
from collections import Counter
import pandas as pd


def atom_counts(smiles):

    mol = Chem.MolFromSmiles(str(smiles))

    if mol is None:
        return {}

    counts = Counter(
        atom.GetSymbol()
        for atom in mol.GetAtoms()
    )

    return counts

In [ ]:
atom_df = (
    df_clean["Canonical_SMILES"]
    .apply(atom_counts)
)

atom_df = pd.DataFrame(
    atom_df.tolist()
).fillna(0)

atom_df = atom_df.astype(int)

In [ ]:
print(atom_df.shape)
print(atom_df.columns)

In [ ]:
X_morgan_plus = pd.concat(
    [
        fp_df.reset_index(drop=True),
        extra_features,
        atom_df.reset_index(drop=True)
    ],
    axis=1
)

In [ ]:
print(X_morgan_plus.shape)

In [ ]:
X_morgan_plus.columns = X_morgan_plus.columns.astype(str)

In [ ]:
rf_morgan_plus = evaluate_random_forest_morgan(
    X_morgan_plus,
    y_clean
)

In [ ]:
print(f"RMSE: {rf_morgan_plus['rmse']:.3f}")
print(f"MAE:  {rf_morgan_plus['mae']:.3f}")
print(f"R²:   {rf_morgan_plus['r2']:.3f}")

In [ ]:
comparison = pd.DataFrame({
    "Model": [
        "RF Bond descriptors",
        "RF Morgan tuned",
        "RF Morgan + charge + MW + atoms"
    ],
    "RMSE": [
        rf_all["rmse"],
        159.683,
        rf_morgan_plus["rmse"]
    ],
    "MAE": [
        rf_all["mae"],
        None,
        rf_morgan_plus["mae"]
    ],
    "R2": [
        rf_all["r2"],
        0.914,
        rf_morgan_plus["r2"]
    ]
})

comparison

In [ ]:
def scan_max_features(X, y, max_features_values):

    results = []

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42
    )

    for mf in max_features_values:

        rf = RandomForestRegressor(
            n_estimators=1000,
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1,
            max_features=mf,
            bootstrap=True,
            random_state=42,
            n_jobs=-1
        )

        rf.fit(X_train, y_train)

        y_pred = rf.predict(X_test)

        results.append({
            "max_features": mf,
            "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
            "MAE": mean_absolute_error(y_test, y_pred),
            "R2": r2_score(y_test, y_pred)
        })

    return pd.DataFrame(results)

In [ ]:
mf_values = [
    "sqrt",
    0.1,
    0.2,
    0.3,
    0.5,
    0.7,
    0.8,
    1.0
]

In [ ]:
results_morgan_plus = scan_max_features(
    X_morgan_plus,
    y_clean,
    mf_values
)

results_morgan_plus

In [ ]:
plt.figure(figsize=(7,5))

plt.plot(
    results_morgan_plus["max_features"].astype(str),
    results_morgan_plus["RMSE"],
    marker="o"
)

plt.xlabel("max_features")
plt.ylabel("RMSE (kJ/mol)")

plt.tight_layout()
plt.show()

## 9. Mordred and ChemML descriptor models


In [ ]:
from chemml.chem import Mordred
from mordred import Calculator, descriptors

In [ ]:
calc = Calculator(
    descriptors,
    ignore_3D=True
)

In [ ]:
mols = [
    Chem.MolFromSmiles(smi)
    for smi in df_clean["Canonical_SMILES"]
]

In [ ]:
import multiprocessing as mp
import time

def calculate_single_mordred(mol):
    try:
        result = calc.pandas(
            [mol],
            nproc=1,
            quiet=True
        )
        return result

    except Exception as e:
        return e


def run_with_timeout(mol, timeout=300):

    with mp.Pool(processes=1) as pool:

        result = pool.apply_async(
            calculate_single_mordred,
            args=(mol,)
        )

        try:
            output = result.get(timeout=timeout)
            return output, True

        except mp.TimeoutError:
            return None, False

In [ ]:
bad_indices = []
mordred_results = []

for i, mol in enumerate(mols):

    print(f"Processing {i}/{len(mols)}")

    result, success = run_with_timeout(
        mol,
        timeout=300
    )

    if success:
        mordred_results.append(result)

    else:
        print(f"Skipping molecule {i}")
        bad_indices.append(i)

In [ ]:
problematic = df_clean.iloc[bad_indices][
    ["species_name", "formula", "Canonical_SMILES"]
]

problematic.to_csv("problematic_molecules.csv", index=True)

problematic

In [ ]:
mordred_df = pd.concat(
    mordred_results,
    ignore_index=True
)

print(mordred_df.shape)

In [ ]:
mordred_df = mordred_df.apply(
    pd.to_numeric,
    errors="coerce"
)

In [ ]:
mordred_df = mordred_df.dropna(axis=1)

In [ ]:
print(mordred_df.shape)

In [ ]:
mordred_df = mordred_df.loc[
    :,
    mordred_df.nunique() > 1
]

In [ ]:
print(mordred_df.shape)

In [ ]:
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

In [ ]:
def rf_max_features_scan(X, y, max_features_values):

    results = []

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42
    )

    for mf in max_features_values:

        rf = RandomForestRegressor(
            n_estimators=1000,
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1,
            max_features=mf,
            bootstrap=True,
            random_state=42,
            n_jobs=-1
        )

        rf.fit(X_train, y_train)

        y_pred = rf.predict(X_test)

        results.append({
            "max_features": mf,
            "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
            "MAE": mean_absolute_error(y_test, y_pred),
            "R2": r2_score(y_test, y_pred)
        })

    return pd.DataFrame(results)

In [ ]:
mf_values = [
    "sqrt",
    0.05,
    0.1,
    0.2,
    0.3,
    0.5,
    0.8,
    1.0
]

In [ ]:
print(mordred_df.shape)
print(df_clean.shape)
print(y_clean.shape)

In [ ]:
df_mordred = df_clean.drop(
    df_clean.index[bad_indices]
).reset_index(drop=True)

y_mordred = df_mordred["dHf_298K"]

In [ ]:
print(mordred_df.shape)
print(df_mordred.shape)
print(y_mordred.shape)

In [ ]:
results_mordred = rf_max_features_scan(
    mordred_df,
    y_mordred,
    mf_values
)

In [ ]:

results_mordred

In [ ]:
X_morgan_mordred = pd.concat(
    [
        fp_df_clean.reset_index(drop=True),
        mordred_df.reset_index(drop=True)
    ],
    axis=1
)

X_morgan_mordred.columns = (
    X_morgan_mordred.columns.astype(str)
)

In [ ]:
results_morgan_mordred = rf_max_features_scan(
    X_morgan_mordred,
    y_mordred,
    mf_values
)

results_morgan_mordred

In [ ]:
bad_indices = problematic.index.tolist()

# Clean every dataframe
fp_df_clean = fp_df.drop(index=bad_indices).reset_index(drop=True)
atom_df_clean = atom_df.drop(index=bad_indices).reset_index(drop=True)
extra_features_clean = extra_features.drop(index=bad_indices).reset_index(drop=True)
df_model = df_clean.drop(index=bad_indices).reset_index(drop=True)

y_model = df_model["dHf_298K"]

# Rename
fp_df_named = fp_df_clean.copy()
fp_df_named.columns = [f"Morgan_{i}" for i in range(fp_df_named.shape[1])]

atom_df_named = atom_df_clean.copy()
atom_df_named.columns = [f"Atom_{c}" for c in atom_df_named.columns]

extra_features_named = extra_features_clean.copy()
extra_features_named.columns = [f"Mol_{c}" for c in extra_features_named.columns]

mordred_df_named = mordred_df.copy()
mordred_df_named.columns = [
    f"Mordred_{i}_{c}"
    for i, c in enumerate(mordred_df_named.columns)
]

# Combine
X_all = pd.concat(
    [
        fp_df_named,
        extra_features_named,
        atom_df_named,
        mordred_df_named
    ],
    axis=1
)

X_all.columns = X_all.columns.astype(str)

print(X_all.shape)
print(y_model.shape)

In [ ]:
results_all = rf_max_features_scan(
    X_all,
    y_model,
    mf_values
)

results_all

In [ ]:
results_mordred["Model"] = "Mordred"
results_morgan_mordred["Model"] = "Morgan + Mordred"
results_all["Model"] = "Morgan + MW + Charge + Atoms + Mordred"


all_results = pd.concat(
    [
        results_mordred,
        results_morgan_mordred,
        results_all
    ],
    ignore_index=True
)

all_results

In [ ]:
plt.figure(figsize=(8,5))

for model in all_results["Model"].unique():

    temp = all_results[
        all_results["Model"] == model
    ]

    plt.plot(
        temp["max_features"].astype(str),
        temp["RMSE"],
        marker="o",
        label=model
    )


plt.xlabel("max_features")
plt.ylabel("RMSE (kJ/mol)")
plt.legend()

plt.tight_layout()
plt.show()

## 10. Extra Trees


In [ ]:
from sklearn.ensemble import ExtraTreesRegressor
def evaluate_extra_trees(X, y, max_features):

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42
    )

    model = ExtraTreesRegressor(
        n_estimators=1000,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        max_features=max_features,
        bootstrap=False,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    return {
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
        "MAE": mean_absolute_error(y_test, y_pred),
        "R2": r2_score(y_test, y_pred),
        "model": model,
        "y_test": y_test,
        "y_pred": y_pred
    }

In [ ]:
mf_values = [
    0.1,
    0.2,
    0.3,
    0.5,
    0.8,
    1.0
]

In [ ]:
extra_results = []

for mf in mf_values:

    result = evaluate_extra_trees(
        X_all,
        y_model,
        mf
    )

    extra_results.append({
        "max_features": mf,
        "RMSE": result["RMSE"],
        "MAE": result["MAE"],
        "R2": result["R2"]
    })


extra_results = pd.DataFrame(extra_results)

extra_results

## 11. XGBoost


In [ ]:
import sys

print(sys.executable)

In [ ]:
import xgboost
print(xgboost.__version__)

In [ ]:
from xgboost import XGBRegressor

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)


def evaluate_xgboost(
    X,
    y,
    params
):

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42
    )


    model = XGBRegressor(
        n_estimators=params["n_estimators"],
        learning_rate=params["learning_rate"],
        max_depth=params["max_depth"],
        subsample=params["subsample"],
        colsample_bytree=params["colsample_bytree"],
        min_child_weight=params["min_child_weight"],
        reg_lambda=params["reg_lambda"],
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )


    model.fit(
        X_train,
        y_train
    )


    y_pred = model.predict(X_test)


    return {
        "model": model,
        "y_test": y_test,
        "y_pred": y_pred,
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
        "MAE": mean_absolute_error(y_test, y_pred),
        "R2": r2_score(y_test, y_pred)
    }

In [ ]:
params_xgb = {
    "n_estimators": 1000,
    "learning_rate": 0.03,
    "max_depth": 6,
    "subsample": 0.8,
    "colsample_bytree": 0.3,
    "min_child_weight": 3,
    "reg_lambda": 1
}

In [ ]:
xgb_result = evaluate_xgboost(
    X_all,
    y_model,
    params_xgb
)
xgb_result